# Demo: Place and Sell a Prediction Market Order

Two things only:
1. **Buy** — place a limit IOC order and confirm the fill
2. **Sell** — sell back the contracts you hold

Set `SANDBOX = True` to use paper trading (no real money).

## 1  Setup

In [ ]:
import os, json
import requests
import pandas as pd

from gemini_trader import GeminiTrader

API_KEY    = os.getenv("GEMINI_API_KEY",    "YOUR_API_KEY_HERE")
API_SECRET = os.getenv("GEMINI_API_SECRET", "YOUR_API_SECRET_HERE")

SANDBOX     = True   # ← False for real money
BET_DOLLARS = 20.0

trader = GeminiTrader(
    api_key       = API_KEY,
    api_secret    = API_SECRET,
    sandbox       = SANDBOX,
    bet_dollars   = BET_DOLLARS,
    cache_ttl_sec = 300,
)

trader.refresh_symbol_cache(force=True)
print(f"Symbol cache: {len(trader._symbol_cache)} contracts loaded")
print(f"Mode: {'SANDBOX' if SANDBOX else 'LIVE'}")

## 2  Pick a contract to trade

In [ ]:
r = requests.get("https://api.gemini.com/v1/prediction-markets/events", timeout=15)
r.raise_for_status()
data   = r.json()
events = data.get("data", data) if isinstance(data, dict) else data

rows = []
for event in events:
    ticker = event.get("ticker", "")
    if not any(x in ticker.upper() for x in ["BTC", "ETH", "SOL"]):
        continue
    for c in event.get("contracts", []):
        if c.get("marketState") != "open":
            continue
        prices = c.get("prices", {})
        rows.append({
            "contract_id":  c.get("id"),
            "label":        c.get("label"),
            "ask_yes":      prices.get("buy",  {}).get("yes"),
            "ask_no":       prices.get("buy",  {}).get("no"),
            "bid_yes":      prices.get("sell", {}).get("yes"),
            "bid_no":       prices.get("sell", {}).get("no"),
        })

df = pd.DataFrame(rows)
print(f"{len(df)} open BTC/ETH/SOL contracts")
df

In [ ]:
# Pick the contract with the cheapest NO (easiest to fill with small notional).
# Change this to any contract_id and side you want.
row = df[df["ask_no"].notna()].sort_values("ask_no").iloc[0]

CONTRACT_ID = row["contract_id"]
SIDE        = "NO"
ASK_PRICE   = float(row["ask_no"])
BID_PRICE   = float(row["bid_no"]) if row["bid_no"] is not None else ASK_PRICE * 0.9

n_contracts = max(1, int(BET_DOLLARS / ASK_PRICE))

print(f"Contract : {row['label']}")
print(f"ID       : {CONTRACT_ID}")
print(f"Side     : {SIDE}")
print(f"Ask      : {ASK_PRICE:.4f}  ({ASK_PRICE*100:.1f}¢)")
print(f"Bid      : {BID_PRICE:.4f}  ({BID_PRICE*100:.1f}¢)")
print(f"Bet size : {n_contracts} contracts × {ASK_PRICE:.4f} = ${n_contracts*ASK_PRICE:.2f}")

## 3  Buy

In [ ]:
buy_result = trader.place_order(
    contract_id = CONTRACT_ID,
    side        = SIDE,
    ask_price   = ASK_PRICE,
    bet_dollars = BET_DOLLARS,
)

print(json.dumps({k: v for k, v in buy_result.items() if k != "raw"}, indent=2))

FILLED = buy_result["filled"]
ORDER_ID = buy_result["order_id"]
print(f"\n→ Filled {FILLED} of {buy_result['n_contracts']} contracts  (order_id={ORDER_ID})")

## 4  Sell

Sells exactly the contracts that were filled in the BUY above.  
Uses the live bid as the limit price.

In [ ]:
if FILLED == 0:
    print("Nothing filled — no sell needed.")
else:
    sell_result = trader.sell_order(
        contract_id = CONTRACT_ID,
        side        = SIDE,
        bid_price   = BID_PRICE,
        n_contracts = FILLED,
    )

    print(json.dumps({k: v for k, v in sell_result.items() if k != "raw"}, indent=2))

    sold = sell_result["filled"]
    pnl  = round((BID_PRICE - ASK_PRICE) * sold, 4)
    print(f"\n→ Sold {sold}/{FILLED} contracts")
    print(f"  Entry ask : {ASK_PRICE:.4f}")
    print(f"  Exit bid  : {BID_PRICE:.4f}")
    print(f"  PnL       : {pnl:+.4f} ({pnl*100:+.2f}¢ total on {sold} contracts)")